# 集群上Delly运行流程

# 一.数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 配对数据有80对，包括N-T配对、P-T配对、P1-T配对、V0227366P-VO227366T配对、V0274654P-VO274654T配对

### -x 排除一些染色体，不进行SV检测，文件路径为："/data/share/RT_group/RT_LAB/huamenglei/SV_tool/Delly/human.hg38.excl.tsv"，上传至集群：/mnt/home/ygjx/chenkejin/delly/delly-main/human.hg38.excl.tsv

### -g 参考基因组，文件路径为："/data/share/PancreaticWGS/Homo_sapiens_assembly38.fasta"，上传至集群：/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta

# 二. 环境配置

In [ ]:
conda create -n delly_env python=3.9 -y
conda activate delly_env
conda install -c bioconda -c conda-forge bcftools=1.23.1 gsl=2.7 -y

bcftools --version  #检查是否安装成功，如果运行报 libgsl.so.25 缺失，则执行以下命令：

cd /mnt/home/ygjx/chenkejin/anaconda3/envs/delly_env/lib/
ln -sf libgsl.so libgsl.so.25
cd /mnt/home/ygjx/chenkejin/delly/delly-main

# 三. 运行

## 1、创建check_pairs.sh检查样本配对情况，代码如下：

In [ ]:
#!/bin/bash

# 定义目标文件夹路径
TARGET_DIR="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
SAMPLES_TSV="$(pwd)/delly_samples.tsv"

echo "========================================="
echo "开始扫描目录: $TARGET_DIR"
echo "========================================="

# 1. 获取目录下的所有项目总数（排除了空行）
ALL_ITEMS=$(ls -1 "$TARGET_DIR")
TOTAL_COUNT=$(echo "$ALL_ITEMS" | sed '/^\s*$/d' | wc -l)

# 2. 分离出符合规则的样本和不符合规则的样本
RECOGNIZED_ITEMS=$(echo "$ALL_ITEMS" | grep -E "(N|P|P1|T)$")
UNRECOGNIZED_ITEMS=$(echo "$ALL_ITEMS" | grep -v -E "(N|P|P1|T)$")
UNRECOGNIZED_COUNT=$(echo "$UNRECOGNIZED_ITEMS" | sed '/^\s*$/d' | wc -l)

# 3. 提取唯一的 Base ID 进行配对
base_ids=$(echo "$RECOGNIZED_ITEMS" | sed -E 's/(N|P1|P|T)$//' | sort | uniq)

paired_count=0
orphan_N_count=0
orphan_T_count=0

> "$SAMPLES_TSV"

echo -e "Base_ID\t\tNormal\t\tTumor(T)\tStatus"
echo "--------------------------------------------------------"

for id in $base_ids; do
    # 跳过空字符串
    if [ -z "$id" ]; then continue; fi

    has_normal=false
    normal_name="-"
    has_tumor=false
    tumor_name="-"
    
    if echo "$RECOGNIZED_ITEMS" | grep -q "^${id}T$"; then 
        has_tumor=true
        tumor_name="${id}T"
    fi
    
    if echo "$RECOGNIZED_ITEMS" | grep -q "^${id}N$"; then 
        has_normal=true
        normal_name="${id}N"
    elif echo "$RECOGNIZED_ITEMS" | grep -q "^${id}P$"; then 
        has_normal=true
        normal_name="${id}P"
    elif echo "$RECOGNIZED_ITEMS" | grep -q "^${id}P1$"; then 
        has_normal=true
        normal_name="${id}P1"
    fi

    if [ "$has_normal" = true ] && [ "$has_tumor" = true ]; then
        echo -e "${id}\t${normal_name}\t\t${tumor_name}\t\t✅ 配对成功"
        ((paired_count++))
        
        echo -e "${normal_name}\tcontrol" >> "$SAMPLES_TSV"
        echo -e "${tumor_name}\ttumor" >> "$SAMPLES_TSV"
        
    elif [ "$has_normal" = true ]; then
        echo -e "${id}\t${normal_name}\t\t-\t\t⚠️ 缺失 Tumor"
        ((orphan_N_count++))
    elif [ "$has_tumor" = true ]; then
        echo -e "${id}\t-\t\t${tumor_name}\t\t⚠️ 缺失 Normal"
        ((orphan_T_count++))
    fi
done

echo "========================================="
echo "                 统计汇总                "
echo "========================================="
echo "📂 目录下总项目数: $TOTAL_COUNT 个"
echo "✅ 成功配对: $paired_count 对 (共 $((paired_count * 2)) 个样本)"
echo "⚠️ 仅有 Normal (N/P/P1): $orphan_N_count 个"
echo "⚠️ 仅有 Tumor (T): $orphan_T_count 个"
echo "-----------------------------------------"

# 4. 重点输出那四个丢失的样本
if [ "$UNRECOGNIZED_COUNT" -gt 0 ]; then
    echo "🚨 发现 $UNRECOGNIZED_COUNT 个未被统计的异常样本 (不以 N, P, P1, T 结尾):"
    echo "$UNRECOGNIZED_ITEMS" | while read -r line; do
        # 排除空行
        if [ -n "$line" ]; then
            echo "   -> $line"
        fi
    done
    echo "💡 请检查它们的命名格式，如果是有效的对照组，请将它们的后缀告诉我，我再加进规则里！"
else
    echo "🎉 所有样本名均符合预期规则，没有漏网之鱼！"
fi
echo "========================================="

## 2、创建 build_task_list.sh 脚本，建立task_list.txt文件，代码如下：

In [ ]:
#!/bin/bash

TARGET_DIR="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
TASK_FILE="task_list.txt"

> "$TASK_FILE"

# 1. 自动提取 78 个正常配堆
# 【核心修复】直接使用 ls -1，完美兼容 HPC 中的软链接(symlink)
ALL_FOLDERS=$(ls -1 "$TARGET_DIR")
RECOGNIZED=$(echo "$ALL_FOLDERS" | grep --color=never -E "(N|P|P1|T)$")
base_ids=$(echo "$RECOGNIZED" | sed -E 's/(N|P1|P|T)$//' | sort | uniq)

for id in $base_ids; do
    if [ -z "$id" ]; then continue; fi
    # 排除那两个名字写错的 Base ID
    if [[ "$id" == "V0227366" || "$id" == "VO227366" || "$id" == "V0274654" || "$id" == "VO274654" ]]; then continue; fi
    
    if echo "$RECOGNIZED" | grep --color=never -q "^${id}T$"; then
        tumor_name="${id}T"
        if echo "$RECOGNIZED" | grep --color=never -q "^${id}N$"; then normal_name="${id}N"
        elif echo "$RECOGNIZED" | grep --color=never -q "^${id}P$"; then normal_name="${id}P"
        elif echo "$RECOGNIZED" | grep --color=never -q "^${id}P1$"; then normal_name="${id}P1"
        else continue
        fi
        echo -e "${normal_name}\t${tumor_name}" >> "$TASK_FILE"
    fi
done

# 2. 手动追加那 2 个特殊配对
echo -e "V0227366P\tVO227366T" >> "$TASK_FILE"
echo -e "V0274654P\tVO274654T" >> "$TASK_FILE"

echo "✅ 任务列表构建完成！共 $(wc -l < "$TASK_FILE") 组配对。"


## 3、创建submit_delly_batch.sh脚本，用来批量提交作业，代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=delly_batch
#SBATCH --output=delly_logs/job_%A_%a.log
#SBATCH --array=1-80%10
#SBATCH --cpus-per-task=16
#SBATCH --mem=60G
#SBATCH --time=72:00:00
#SBATCH --export=ALL

# ================= 1. 基础路径配置 =================
WORK_DIR="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
OUT_DIR="/mnt/home/ygjx/chenkejin/delly_results"
LOG_DIR="delly_logs"
TASK_LIST="task_list.txt"
SUMMARY_LOG="overall_progress.log"

REF_FASTA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"      
EXCL_FILE="/mnt/home/ygjx/chenkejin/delly/delly-main/human.hg38.excl.tsv" 

mkdir -p "$OUT_DIR"
mkdir -p "$LOG_DIR"

# ================= 2. 读取当前样本信息 =================
# 注意：确保 task_list.txt 每一行格式为: Normal_ID Tumor_ID
TASK_INFO=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST")
NORMAL_ID=$(echo "$TASK_INFO" | awk '{print $1}')
TUMOR_ID=$(echo "$TASK_INFO" | awk '{print $2}')

if [ -z "$NORMAL_ID" ] || [ -z "$TUMOR_ID" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] ❌ 任务读取失败，跳过。"
    exit 1
fi

# 记录任务启动到全局日志
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 🚀 任务启动: ArrayID_${SLURM_ARRAY_TASK_ID} (${NORMAL_ID} vs ${TUMOR_ID})" >> "$SUMMARY_LOG"
echo "--------------------------------------------------------"
echo "开始处理样本对: ${NORMAL_ID} (Normal) vs ${TUMOR_ID} (Tumor)"
echo "--------------------------------------------------------"

# ================= 3. 断点续传检查 =================
FINAL_VCF="${OUT_DIR}/${TUMOR_ID}_vs_${NORMAL_ID}_somatic.vcf"
if [ -f "$FINAL_VCF" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] ✅ 最终结果已存在，跳过该样本对: $FINAL_VCF" >> "$SUMMARY_LOG"
    exit 0
fi

# ================= 4. 定义真实文件路径 =================
REAL_NORMAL_BAM="${WORK_DIR}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
REAL_NORMAL_BAI="${WORK_DIR}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bai"
REAL_TUMOR_BAM="${WORK_DIR}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"
REAL_TUMOR_BAI="${WORK_DIR}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bai"

# ================= 5. 构建专属隔离沙盒 =================
SANDBOX_DIR="${OUT_DIR}/sandbox_${NORMAL_ID}_${TUMOR_ID}"
mkdir -p "$SANDBOX_DIR"
cd "$SANDBOX_DIR" || exit 1

echo "🔗 正在创建环境软链接..."
ln -sf "$REAL_NORMAL_BAM" "local_${NORMAL_ID}.bam"
ln -sf "$REAL_NORMAL_BAI" "local_${NORMAL_ID}.bai"
ln -sf "$REAL_TUMOR_BAM" "local_${TUMOR_ID}.bam"
ln -sf "$REAL_TUMOR_BAI" "local_${TUMOR_ID}.bai"

TMP_TSV="samples_${NORMAL_ID}_${TUMOR_ID}.tsv"
echo -e "${NORMAL_ID}\tcontrol\n${TUMOR_ID}\ttumor" > "$TMP_TSV"

# ================= 6. 执行 Delly Pipeline =================
RAW_BCF="${TUMOR_ID}_vs_${NORMAL_ID}_raw.bcf"
FILTERED_BCF="${TUMOR_ID}_vs_${NORMAL_ID}_somatic.bcf"

echo "⏳ 执行 delly call..."
delly call -x "$EXCL_FILE" -g "$REF_FASTA" -o "$RAW_BCF" "local_${TUMOR_ID}.bam" "local_${NORMAL_ID}.bam"

echo "⏳ 执行 delly filter..."
delly filter -f somatic -o "$FILTERED_BCF" -s "$TMP_TSV" "$RAW_BCF"

echo "⏳ 转换为 VCF 格式..."
bcftools view "$FILTERED_BCF" > "$FINAL_VCF"

# ================= 7. 任务结束记录 =================
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 🎉 任务 ArrayID_${SLURM_ARRAY_TASK_ID} (${NORMAL_ID} vs ${TUMOR_ID}) 完美结束！" >> "$SUMMARY_LOG"
echo "✅ 结果已生成: $FINAL_VCF"null

echo "[$(date '+%Y-%m-%d %H:%M:%S')] 🎉 任务 $NORMAL_ID & $TUMOR_ID 完美结束！"

## 输入 sbatch submit_delly_batch.sh 提交作业，输入 tail -f overall_progress.log 可查看整体进度

## 运行结果路径： /mnt/home/ygjx/chenkejin/delly_results/